# Member A -- Learning Rate Experiments

My 10 runs for our hyperparameter sweep. I'm only touching `learning_rate`
here -- everything else stays at the baseline we agreed on in
`shared_train.py`, so our results are actually comparable across all
three of us.

In [ ]:
!pip install -q "stable-baselines3[extra]" "gymnasium[atari,accept-rom-license]" ale-py "autorom[accept-rom-license]" pandas
!AutoROM --accept-license

## Pull our repo

This grabs `shared_train.py` and everything else from our repo so the
import below works.

In [ ]:
import os

REPO_DIR = "/content/deep-q-learning-formative3"
if os.path.exists(REPO_DIR):
    # reset --hard (not pull) so this always matches the remote exactly,
    # even if history upstream was rewritten (force-pushed) since your
    # last clone -- a plain pull can't reconcile that.
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} reset --hard origin/main
else:
    !git clone https://github.com/Mikekimm/deep-q-learning-formative3.git {REPO_DIR}
%cd {REPO_DIR}

from shared_train import BASELINE_CONFIG, GAME_ID, TOTAL_TIMESTEPS, SEED, train_one_run
print(GAME_ID, TOTAL_TIMESTEPS, SEED)
print(BASELINE_CONFIG)

## Smoke test -- run this first

Same pipeline as a real run, just 2,000 steps instead of 200,000 so it
only takes a minute or two. Run it by itself first to make sure
everything's actually working in this session before committing hours
to the real experiments below (don't Run All the whole notebook).

In [ ]:
import shared_train

_original_total_timesteps = shared_train.TOTAL_TIMESTEPS
shared_train.TOTAL_TIMESTEPS = 2000  # temporary override for this test only

smoke_row = shared_train.train_one_run(overrides={}, run_name="smoketest", member="test")
print(smoke_row)

shared_train.TOTAL_TIMESTEPS = _original_total_timesteps  # restore before real runs below
print("TOTAL_TIMESTEPS restored to", shared_train.TOTAL_TIMESTEPS)

In [ ]:
# 10 learning_rate values spanning the useful range; everything else
# stays at BASELINE_CONFIG.
lr_values = [1e-2, 5e-3, 1e-3, 7e-4, 5e-4, 3e-4, 1e-4, 7e-5, 5e-5, 1e-5]

## Optional: live reward graph

Run this before the loop below if you want to watch reward trends
update live instead of waiting for a run to finish. It's just the
reward graph, not actual gameplay -- we're not rendering the game
during training since it'd slow everything down.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir results/tb_logs

In [ ]:
results = []
for i, lr in enumerate(lr_values, start=1):
    row = train_one_run(
        overrides={"learning_rate": lr},
        run_name=f"memberA_lr_{i:02d}",
        member="MemberA",
    )
    results.append(row)